In [ ]:
import logging
import boto3
import json
import pandas as pd
from botocore.exceptions import ClientError

Crea un notebook flights_analytics.ipynb en tu repositorio. El notebook debe:

Conectarse a la Read Replica con SQLAlchemy y pd.read_sql

Ejecutar las 8 preguntas (P1–P5 y W1–W3) como queries SQL

Mostrar el resultado de cada query como DataFrame de pandas

Incluir al menos una visualización por pregunta usando cualquiera de las siguientes librerías:

matplotlib — gráficas estáticas (plt.style.use('ggplot'))
plotnine — gramática de gráficas estilo ggplot2
plotly — gráficas interactivas
great_tables — tablas formateadas con resaltado, colores y barras inline
Elige la herramienta que mejor comunique la respuesta de cada pregunta: barras para rankings (P1, P2, P5), tablas formateadas para conteos (P3), líneas para series de tiempo (P4, W2), y tablas con ranking para W1 y W3.

In [ ]:
# ──────────────────────────────────────────────
# Configuración del logger
# ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

In [ ]:
def get_secret():

    secret_name = "itam/rds/flights/credentials"
    region_name = "us-east-1"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        logger.error(f"Error retrieving secret: {e}")
        raise e

    secret = get_secret_value_response['SecretString']
    return json.loads(secret)

creds   = get_secret()

In [ ]:
#conectarse a la base de datos con SQLAlchemy
from sqlalchemy import create_engine

# RDS endpoint Replica
RDS_ENDPOINT = "itam-flights-replica-956463123241.cmvsysimg4pq.us-east-1.rds.amazonaws.com"
DB_NAME      = creds["dbname"]
DB_USER      = creds["username"]
DB_PASSWORD  = creds["password"]
DB_PORT      = creds["port"]

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{RDS_ENDPOINT}:{DB_PORT}/{DB_NAME}"
)
logger.info("Conexión a la base de datos establecida con SQLAlchemy")

# Verificar conexión
try:
    with engine.connect() as conn:
        print("✓ Conexión exitosa a RDS PostgreSQL")
except SQLAlchemyError as e:
    print(f"✗ Error de conexión: {e}")

In [ ]:
# ──────────────────────────────────────────────
# Consultas de prueba
# ──────────────────────────────────────────────
query_1 = "SELECT COUNT(*) FROM airlines;"